# Dual-Branch PQ-FiLM Stack QNN  ·  A6000-Optimised
## Prior-Guided Quantum FiLM Fusion · 4-Class Stack Classification + Gram Regression

**Architecture:**
- **Branch 1** — Dense MLP on PCA(256) IQ features → 64-d
- **Branch 2** — Pre-extracted EfficientNet-B0 features (1280-d, cached) → projection → 64-d
- **Quantum** — AngleEmbedding VQC (4 qubits, `lightning.gpu`) on Branch-1 input
- **PQ-FiLM Fusion** — quantum independently modulates both branches, cross-branch gate
- **Dual Head** — 4-class softmax + gram regression

**A6000 speed tricks:**
- All training tensors pre-loaded to GPU (fits in 48 GB; eliminates CPU→GPU transfers per batch)
- AMP `autocast` + `GradScaler` on classical layers
- `batch_size=256`, `cudnn.benchmark=True`
- EfficientNet pre-extraction in batches of 512 (one-time, then cached)

In [18]:
import os, sys, time, json, warnings, random, copy
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from scipy.signal import get_window
from scipy.stats import skew, kurtosis as scipy_kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, r2_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.models as tv_models

import pennylane as qml

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

warnings.filterwarnings('ignore')

SEED = 42
def set_seed(s):
    os.environ['PYTHONHASHSEED'] = str(s)
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = False    # False for speed
    torch.backends.cudnn.benchmark     = True     # auto-tune kernels for fixed input sizes
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('Device: CPU')

Device: NVIDIA RTX A6000
VRAM  : 51.0 GB


In [19]:
BASE_DIR = Path('/home/sammarv/quantum_corrosion')

TARGET_CLASSES = [
    {'id': 0, 'name': 'Stack 1.0g',  'path_sub': 'stack/1/1gSTACK',       'gram': 1.0},
    {'id': 1, 'name': 'Stack 1.5g',  'path_sub': 'stack/1.5/1.5gSTACK',   'gram': 1.5},
    {'id': 2, 'name': 'Stack 2.0g',  'path_sub': 'stack/2/2gSTACK',       'gram': 2.0},
    {'id': 3, 'name': 'Stack 2.5g',  'path_sub': 'stack/2.5/2.5gSTACK',   'gram': 2.5},
]

N_CLASSES   = len(TARGET_CLASSES)
GRAM_VALUES = {i: tc['gram'] for i, tc in enumerate(TARGET_CLASSES)}
LABEL_NAMES = [tc['name'] for tc in TARGET_CLASSES]

OUT_DIR = BASE_DIR / 'results/stack_pqfilm_dual'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FFT_SIZE        = 4096
N_STACKS        = 16
OVERLAP         = 0.25
HOP             = int(FFT_SIZE * (1 - OVERLAP))
SAMPLES_PER_IMG = N_STACKS * HOP + (FFT_SIZE - HOP)
WIN             = get_window('hann', FFT_SIZE).astype(np.float32)

TRAIN_FRAC = 0.60
VAL_FRAC   = 0.20
N_WORKERS  = 16    # CPU threads for IQ feature extraction
CNN_BATCH  = 512   # A6000 can process 512 spectrograms per EfficientNet pass

print(f'SAMPLES_PER_IMG: {SAMPLES_PER_IMG:,}')
for i, tc in enumerate(TARGET_CLASSES):
    p  = BASE_DIR / tc['path_sub']
    iq = np.memmap(str(p), dtype='complex64', mode='r')
    print(f'  label={i}  {tc["name"]:14s}  images={len(iq)//SAMPLES_PER_IMG:,}')

SAMPLES_PER_IMG: 50,176
  label=0  Stack 1.0g      images=8,484
  label=1  Stack 1.5g      images=9,340
  label=2  Stack 2.0g      images=9,158
  label=3  Stack 2.5g      images=9,339


In [20]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. FEATURE EXTRACTION — dense IQ (1040-d) + spectrogram (16×512)
# ═══════════════════════════════════════════════════════════════════════════════

def extract_one_dense(iq, img_idx):
    start = img_idx * SAMPLES_PER_IMG
    end   = start + SAMPLES_PER_IMG
    if end > len(iq): return None
    frames = np.empty((N_STACKS, FFT_SIZE), dtype='complex64')
    for i in range(N_STACKS):
        s = start + i * HOP
        frames[i] = iq[s : s + FFT_SIZE]
    spec     = np.fft.fftshift(np.fft.fft(frames * WIN, axis=1), axes=1)
    mag      = np.abs(spec).astype(np.float32)
    psd_mean = mag.mean(axis=0)
    psd_512  = (20.0 * np.log10(psd_mean + 1e-12)).reshape(512, 8).mean(axis=1)
    var_512  = np.log1p(mag.var(axis=0).reshape(512, 8).mean(axis=1))
    freqs    = np.arange(FFT_SIZE, dtype=np.float32)
    centroid = float((freqs * psd_mean).sum() / (psd_mean.sum() + 1e-12))
    fp       = mag.sum(axis=1)
    stats    = np.array([
        float(psd_mean.sum()), centroid,
        float(np.sqrt(((freqs-centroid)**2 * psd_mean).sum() / (psd_mean.sum()+1e-12))),
        float(np.exp(np.log(psd_mean+1e-12).mean()) / (psd_mean.mean()+1e-12)),
        float(-(psd_mean/(psd_mean.sum()+1e-12)*np.log(psd_mean/(psd_mean.sum()+1e-12)+1e-12)).sum()),
        float(fp.var()), float(skew(fp)), float(scipy_kurtosis(fp)),
    ], dtype=np.float32)
    phase     = np.angle(spec).astype(np.float32)
    inst_freq = np.diff(np.unwrap(phase, axis=0), axis=0)
    i_pwr = (frames.real**2).mean(); q_pwr = (frames.imag**2).mean()
    amp   = np.abs(frames.flatten()[:4096])
    phase_feats = np.array([
        float(np.std(phase.mean(axis=0))), float(phase.var(axis=1).mean()),
        float(inst_freq.mean()), float(inst_freq.std()),
        float(i_pwr / (q_pwr+1e-12)),
        float(np.corrcoef(frames.real.flatten()[:2048], frames.imag.flatten()[:2048])[0,1]),
        float(scipy_kurtosis(amp)), float(skew(amp)),
    ], dtype=np.float32)
    return np.concatenate([psd_512, var_512, stats, phase_feats])


def extract_one_spec(iq, img_idx):
    start = img_idx * SAMPLES_PER_IMG
    end   = start + SAMPLES_PER_IMG
    if end > len(iq): return None
    frames = np.empty((N_STACKS, FFT_SIZE), dtype='complex64')
    for i in range(N_STACKS):
        s = start + i * HOP
        frames[i] = iq[s : s + FFT_SIZE]
    spec    = np.fft.fftshift(np.fft.fft(frames * WIN, axis=1), axes=1)
    mag_512 = np.abs(spec).astype(np.float32).reshape(N_STACKS, 512, 8).mean(axis=2)
    spec_db = 20.0 * np.log10(mag_512 + 1e-12)
    mn, mx  = spec_db.min(), spec_db.max()
    return ((spec_db - mn) / (mx - mn + 1e-12)).astype(np.float32)


def extract_split_dual(records, iqs, desc=''):
    n = len(records)
    D, S, L = [None]*n, [None]*n, [None]*n
    t0 = time.time()
    def _w(i, lbl, idx):
        return i, extract_one_dense(iqs[lbl], idx), extract_one_spec(iqs[lbl], idx), lbl
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(_w, i, lbl, idx): i for i, (lbl, idx) in enumerate(records)}
        done = 0
        for fut in as_completed(futs):
            i, d, s, lbl = fut.result()
            D[i], S[i], L[i] = d, s, lbl
            done += 1
            if done % 1000 == 0 or done == n:
                e = time.time() - t0
                print(f'  [{desc}] {done}/{n}  {e:.0f}s  ETA={e/done*(n-done):.0f}s')
    valid = [(d,s,l) for d,s,l in zip(D,S,L) if d is not None and s is not None]
    return (np.stack([v[0] for v in valid]),
            np.stack([v[1] for v in valid]),
            np.array([v[2] for v in valid]))

print('Feature extraction ready.')

Feature extraction ready.


In [21]:
rng = np.random.default_rng(SEED)
iqs = {i: np.memmap(str(BASE_DIR / tc['path_sub']), dtype='complex64', mode='r')
       for i, tc in enumerate(TARGET_CLASSES)}

train_records, val_records, test_records = [], [], []
for lbl, iq in iqs.items():
    n      = len(iq) // SAMPLES_PER_IMG
    idx    = rng.permutation(n)
    n_tr   = int(n * TRAIN_FRAC)
    n_val  = int(n * VAL_FRAC)
    train_records.extend((lbl, int(i)) for i in idx[:n_tr])
    val_records.extend((lbl, int(i))   for i in idx[n_tr:n_tr+n_val])
    test_records.extend((lbl, int(i))  for i in idx[n_tr+n_val:])
    print(f'  {TARGET_CLASSES[lbl]["name"]}:  train={n_tr}  val={n_val}  test={n-n_tr-n_val}')

rng.shuffle(train_records); rng.shuffle(val_records); rng.shuffle(test_records)
print(f'Total: train={len(train_records)}  val={len(val_records)}  test={len(test_records)}')

  Stack 1.0g:  train=5090  val=1696  test=1698
  Stack 1.5g:  train=5604  val=1868  test=1868
  Stack 2.0g:  train=5494  val=1831  test=1833
  Stack 2.5g:  train=5603  val=1867  test=1869
Total: train=21791  val=7262  test=7268


In [22]:
cache_dir = OUT_DIR / 'feature_cache'
cache_dir.mkdir(exist_ok=True)

def load_or_extract(records, split_name):
    cd = cache_dir / f'{split_name}_X_dense.npy'
    cs = cache_dir / f'{split_name}_X_spec.npy'
    cy = cache_dir / f'{split_name}_y.npy'
    if cd.exists() and cs.exists() and cy.exists():
        print(f'  [{split_name}] loaded from cache')
        return np.load(cd), np.load(cs), np.load(cy)
    print(f'  [{split_name}] extracting {len(records)} samples...')
    X_d, X_s, y = extract_split_dual(records, iqs, desc=split_name)
    np.save(cd, X_d); np.save(cs, X_s); np.save(cy, y)
    return X_d, X_s, y

X_tr_d,  X_tr_s,  y_tr  = load_or_extract(train_records, 'train')
X_val_d, X_val_s, y_val = load_or_extract(val_records,   'val')
X_te_d,  X_te_s,  y_te  = load_or_extract(test_records,  'test')

for arr in (X_tr_d, X_val_d, X_te_d):
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Dense {X_tr_d.shape}  Spec {X_tr_s.shape}')

  [train] loaded from cache
  [val] loaded from cache
  [test] loaded from cache
Dense (21791, 1040)  Spec (21791, 16, 512)


In [23]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRE-EXTRACT EFFICIENTNET-B0 FEATURES  (runs ONCE, then cached)
# A6000: CNN_BATCH=512 keeps GPU ~fully utilised during extraction.
# After this cell EfficientNet is deleted; it never touches the training loop.
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def preextract_cnn(X_spec, split_name):
    cache_path = cache_dir / f'{split_name}_X_cnn.npy'
    if cache_path.exists():
        print(f'  [{split_name}] CNN cache hit')
        return np.load(cache_path)

    print(f'  [{split_name}] extracting CNN features ({len(X_spec)} samples)...')
    eff      = tv_models.efficientnet_b0(
                   weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1
               ).to(DEVICE).eval()
    backbone = eff.features
    pool     = nn.AdaptiveAvgPool2d(1)
    out = []
    t0  = time.time()
    for start in range(0, len(X_spec), CNN_BATCH):
        chunk = X_spec[start : start + CNN_BATCH]                  # (B,16,512)
        t = torch.from_numpy(chunk).unsqueeze(1)                   # (B,1,16,512)
        t = F.interpolate(t, size=(224,224), mode='bilinear',
                          align_corners=False).repeat(1,3,1,1)     # (B,3,224,224)
        with torch.amp.autocast('cuda'):
            feats = pool(backbone(t.to(DEVICE))).flatten(1).float() # (B,1280) fp32
        out.append(feats.cpu().numpy())
        done = min(start + CNN_BATCH, len(X_spec))
        if done % (CNN_BATCH*8) == 0 or done == len(X_spec):
            print(f'    {done}/{len(X_spec)}  {time.time()-t0:.0f}s')

    del eff, backbone, pool
    torch.cuda.empty_cache()

    result = np.concatenate(out, axis=0).astype(np.float32)
    np.save(cache_path, result)
    print(f'  [{split_name}] saved {result.shape} → {cache_path}')
    return result


print('Pre-extracting EfficientNet features...')
X_tr_cnn  = preextract_cnn(X_tr_s,  'train')
X_val_cnn = preextract_cnn(X_val_s, 'val')
X_te_cnn  = preextract_cnn(X_te_s,  'test')
print(f'CNN shapes — train={X_tr_cnn.shape}  val={X_val_cnn.shape}  test={X_te_cnn.shape}')
del X_tr_s, X_val_s, X_te_s    # free RAM

Pre-extracting EfficientNet features...
  [train] CNN cache hit
  [val] CNN cache hit
  [test] CNN cache hit
CNN shapes — train=(21791, 1280)  val=(7262, 1280)  test=(7268, 1280)


In [24]:
# ─── Branch 1: StandardScaler + PCA(256) ──────────────────────────────────────
scaler = StandardScaler()
pca    = PCA(n_components=256, random_state=SEED)
X_tr_pca  = pca.fit_transform(scaler.fit_transform(X_tr_d)).astype(np.float32)
X_val_pca = pca.transform(scaler.transform(X_val_d)).astype(np.float32)
X_te_pca  = pca.transform(scaler.transform(X_te_d)).astype(np.float32)
print(f'PCA 256 explains {pca.explained_variance_ratio_.cumsum()[-1]*100:.1f}% of variance')

# ─── Branch 2: StandardScaler on CNN features ─────────────────────────────────
cnn_scaler  = StandardScaler()
X_tr_cnn_s  = cnn_scaler.fit_transform(X_tr_cnn).astype(np.float32)
X_val_cnn_s = cnn_scaler.transform(X_val_cnn).astype(np.float32)
X_te_cnn_s  = cnn_scaler.transform(X_te_cnn).astype(np.float32)

# ─── Class weights ─────────────────────────────────────────────────────────────
counts = Counter(y_tr.tolist())
cw = torch.tensor([1.0/counts[c] for c in range(N_CLASSES)],
                   dtype=torch.float32, device=DEVICE)
cw = cw / cw.sum() * N_CLASSES
print('Class weights:', [f'{w:.4f}' for w in cw.tolist()])

# ─── Estimate memory footprint ─────────────────────────────────────────────────
total_mb = sum(a.nbytes for a in [
    X_tr_pca, X_val_pca, X_te_pca,
    X_tr_cnn_s, X_val_cnn_s, X_te_cnn_s
]) / 1e6
print(f'All training tensors: {total_mb:.0f} MB  (will be pre-loaded to GPU)')

PCA 256 explains 43.6% of variance
Class weights: ['1.0686', '0.9706', '0.9900', '0.9708']
All training tensors: 223 MB  (will be pre-loaded to GPU)


In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# GPU-RESIDENT DATASET
# Moves all tensors to GPU once at construction time.
# __getitem__ is a pure GPU tensor slice — zero CPU↔GPU transfer per batch.
# ═══════════════════════════════════════════════════════════════════════════════

class GPUDataset(Dataset):
    def __init__(self, X_pca, X_cnn, y):
        self.x1 = torch.from_numpy(X_pca).to(DEVICE)
        self.x2 = torch.from_numpy(X_cnn).to(DEVICE)
        self.y  = torch.tensor(y, dtype=torch.long, device=DEVICE)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x1[idx], self.x2[idx], self.y[idx]


def make_loaders(bs):
    # num_workers=0: data is already on GPU, multiprocessing would be counterproductive
    # pin_memory=False: tensors are already in CUDA memory
    kw = dict(num_workers=0, pin_memory=False)
    tr  = DataLoader(GPUDataset(X_tr_pca,  X_tr_cnn_s,  y_tr),  batch_size=bs, shuffle=True,  drop_last=True,  **kw)
    val = DataLoader(GPUDataset(X_val_pca, X_val_cnn_s, y_val), batch_size=bs, shuffle=False, drop_last=False, **kw)
    te  = DataLoader(GPUDataset(X_te_pca,  X_te_cnn_s,  y_te),  batch_size=bs, shuffle=False, drop_last=False, **kw)
    return tr, val, te


_ds = GPUDataset(X_tr_pca[:4], X_tr_cnn_s[:4], y_tr[:4])
x1, x2, yy = _ds[0]
print(f'x_dense: {x1.shape}  device={x1.device}')
print(f'x_cnn  : {x2.shape}  device={x2.device}')
del _ds

x_dense: torch.Size([256])  device=cuda:0
x_cnn  : torch.Size([1280])  device=cuda:0


In [26]:
def _make_device(n_qubits):
    if torch.cuda.is_available():
        try:
            return qml.device('lightning.gpu', wires=n_qubits)
        except Exception:
            pass
    try:
        return qml.device('lightning.qubit', wires=n_qubits)
    except Exception:
        return qml.device('default.qubit', wires=n_qubits)

_test_dev = _make_device(4)
print(f'Quantum device: {_test_dev.name}')

Quantum device: lightning.gpu


In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedPriorGuidedQuantumFiLM(nn.Module):
    """Quantum output q modulates c1 and c2 independently; cross-gate blends them."""
    def __init__(self, c_dim=64, q_dim=4):
        super().__init__()
        self.film1     = nn.Sequential(nn.Linear(q_dim,32), nn.SiLU(), nn.LayerNorm(32), nn.Linear(32,c_dim*3))
        self.gate1     = nn.Sequential(nn.Linear(c_dim+q_dim, c_dim), nn.LayerNorm(c_dim))
        self.film2     = nn.Sequential(nn.Linear(q_dim,32), nn.SiLU(), nn.LayerNorm(32), nn.Linear(32,c_dim*3))
        self.gate2     = nn.Sequential(nn.Linear(c_dim+q_dim, c_dim), nn.LayerNorm(c_dim))
        self.cross     = nn.Sequential(nn.Linear(c_dim*2+q_dim, c_dim), nn.SiLU(), nn.Linear(c_dim,1))
        self.q_proj    = nn.Sequential(nn.Linear(q_dim,16), nn.SiLU())
        feat = c_dim + 16
        self.classifier = nn.Sequential(nn.Linear(feat,64), nn.SiLU(), nn.Dropout(0.3), nn.Linear(64,N_CLASSES))
        self.regressor  = nn.Sequential(nn.Linear(feat,32), nn.SiLU(), nn.Dropout(0.3), nn.Linear(32,1))

    def forward(self, c1, c2, q):
        g1,b1,a1 = torch.chunk(self.film1(q),3,dim=1)
        c1m = c1*(g1+1)+b1
        f1  = torch.sigmoid(self.gate1(torch.cat([c1m,q],1))+a1)*c1m + (1-torch.sigmoid(self.gate1(torch.cat([c1m,q],1))+a1))*c1
        g2,b2,a2 = torch.chunk(self.film2(q),3,dim=1)
        c2m = c2*(g2+1)+b2
        f2  = torch.sigmoid(self.gate2(torch.cat([c2m,q],1))+a2)*c2m + (1-torch.sigmoid(self.gate2(torch.cat([c2m,q],1))+a2))*c2
        w   = torch.sigmoid(self.cross(torch.cat([f1,f2,q],1)))
        feat = torch.cat([w*f1+(1-w)*f2, self.q_proj(q)], dim=1)
        return self.classifier(feat), self.regressor(feat)


class DualBranchPQFiLMQNN(nn.Module):
    """
    x1 : (B, 256)  — PCA IQ features  (Branch 1)
    x2 : (B, 1280) — cached EfficientNet features  (Branch 2)
    """
    def __init__(self, n_qubits, n_layers, ansatz, angle_scaling, dropout_c):
        super().__init__()
        self.angle_scaling = angle_scaling

        self.branch1 = nn.Sequential(
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_c),
            nn.Linear(128,64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout_c))

        self.branch2 = nn.Sequential(
            nn.Linear(1280,256), nn.BatchNorm1d(256), nn.SiLU(), nn.Dropout(dropout_c),
            nn.Linear(256,64),   nn.BatchNorm1d(64),  nn.SiLU(), nn.Dropout(dropout_c))

        self.qnn_proj = nn.Sequential(nn.Linear(256, n_qubits), nn.Sigmoid())
        dev = _make_device(n_qubits)
        diff = 'adjoint' if 'lightning' in dev.name else 'backprop'

        @qml.qnode(dev, interface='torch', diff_method=diff)
        def _qnode(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits))
            if ansatz == 'StronglyEntangling':
                qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            else:
                qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        w_shape = ({'weights': (n_layers, n_qubits, 3)}
                   if ansatz == 'StronglyEntangling'
                   else {'weights': (n_layers, n_qubits)})
        self.qnn    = qml.qnn.TorchLayer(_qnode, w_shape)
        self.fusion = EnhancedPriorGuidedQuantumFiLM(c_dim=64, q_dim=n_qubits)

    def forward(self, x1, x2):
        c1    = self.branch1(x1)
        c2    = self.branch2(x2)
        q_out = self.qnn(self.qnn_proj(x1) * self.angle_scaling)
        logits, grams = self.fusion(c1, c2, q_out)
        return logits, grams.squeeze(1)

print('Model classes defined.')

Model classes defined.


In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES  (AMP on classical layers; QNN stays in float32)
# ═══════════════════════════════════════════════════════════════════════════════

scaler_amp = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

def train_epoch(model, loader, opt, clf_fn, reg_fn, alpha=0.3):
    model.train()
    total_loss = correct = n = 0
    for x1_b, x2_b, y_b in loader:
        # Data already on GPU — no .to() needed
        g_b = torch.tensor([GRAM_VALUES[int(l)] for l in y_b.cpu()],
                            dtype=torch.float32, device=DEVICE)
        opt.zero_grad(set_to_none=True)   # faster than zero_grad()

        # Classical branches under AMP; QNN projection and QNN itself in fp32
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            c1   = model.branch1(x1_b)
            c2   = model.branch2(x2_b)
            q_in = (model.qnn_proj(x1_b) * model.angle_scaling).float()  # fp32 for QNN

        q_out = model.qnn(q_in)   # QNN always in fp32

        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            logits, grams = model.fusion(c1.float(), c2.float(), q_out)
            loss = ((1-alpha) * clf_fn(logits, y_b)
                    + alpha   * reg_fn(grams.float(), g_b))

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(opt)
        scaler_amp.update()

        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        n          += len(y_b)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues, gpreds, gtrues = [], [], [], []
    for x1_b, x2_b, y_b in loader:
        logits, grams = model(x1_b, x2_b)
        preds.extend(logits.argmax(1).cpu().tolist())
        trues.extend(y_b.cpu().tolist())
        gpreds.extend(grams.cpu().tolist())
        gtrues.extend([GRAM_VALUES[int(l)] for l in y_b.cpu()])
    return (
        accuracy_score(trues, preds),
        f1_score(trues, preds, average='macro', zero_division=0),
        mean_absolute_error(gtrues, gpreds),
        r2_score(gtrues, gpreds),
        preds, trues
    )

print('Training utilities ready.')

Training utilities ready.


In [29]:
ANGLE_MAP = {'pi_half': np.pi/2, 'pi': np.pi, 'two_pi': 2*np.pi}

best_params = {
    'n_qubits':      4,
    'n_layers':      2,
    'ansatz':        'StronglyEntangling',
    'angle_scaling': 'pi',
    'lr':             8e-4,
    'weight_decay':   1e-4,
    'dropout_c':      0.2,
    'batch_size':     256,   # A6000: large batch, data already on GPU
}

WARMUP_EPOCHS = 10
FINAL_EPOCHS  = 100
PATIENCE      = 20
ALPHA         = 0.3

for k, v in best_params.items(): print(f'  {k}: {v}')

  n_qubits: 4
  n_layers: 2
  ansatz: StronglyEntangling
  angle_scaling: pi
  lr: 0.0008
  weight_decay: 0.0001
  dropout_c: 0.2
  batch_size: 256


In [30]:
tr_loader, val_loader, te_loader = make_loaders(best_params['batch_size'])

model = DualBranchPQFiLMQNN(
    n_qubits      = best_params['n_qubits'],
    n_layers      = best_params['n_layers'],
    ansatz        = best_params['ansatz'],
    angle_scaling = ANGLE_MAP[best_params['angle_scaling']],
    dropout_c     = best_params['dropout_c'],
).to(DEVICE)

clf_loss = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
reg_loss = nn.HuberLoss(delta=0.5)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Smoke test
model.eval()
with torch.no_grad():
    _x1, _x2, _ = next(iter(tr_loader))
    _lg, _gr = model(_x1[:2], _x2[:2])
    print(f'Forward OK — logits={_lg.shape}  grams={_gr.shape}')

Trainable params: 426,546
Forward OK — logits=torch.Size([2, 4])  grams=torch.Size([2])


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TWO-PHASE TRAINING
# Phase 1 (WARMUP_EPOCHS) : QNN + classical
# Phase 2                 : QNN frozen → classical only at full GPU speed
# ═══════════════════════════════════════════════════════════════════════════════

def make_opt(m, lr, wd):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, m.parameters()), lr=lr, weight_decay=wd)

opt       = make_opt(model, best_params['lr'], best_params['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)

best_val_f1  = 0.0
best_weights = None
no_improve   = 0
phase        = 1
history      = {'tr_acc':[], 'val_f1':[], 'val_mae':[], 'val_r2':[], 'phase':[]}

print(f'max={FINAL_EPOCHS} epochs  warmup={WARMUP_EPOCHS}  patience={PATIENCE}  bs={best_params["batch_size"]}')
print(f'{"Ep":>4}  {"Ph":>2}  {"TrAcc":>7}  {"ValF1":>7}  {"MAE":>6}  {"R2":>6}  {"Best":>7}  Wait')

for ep in range(1, FINAL_EPOCHS+1):
    if ep == WARMUP_EPOCHS+1 and phase == 1:
        for p in model.qnn.parameters(): p.requires_grad_(False)
        opt       = make_opt(model, best_params['lr'], best_params['weight_decay'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
        phase = 2
        print(f'[ep {ep}] QNN frozen → classical-only')

    tr_loss, tr_acc = train_epoch(model, tr_loader, opt, clf_loss, reg_loss, ALPHA)
    val_acc, val_f1, val_mae, val_r2, _, _ = evaluate(model, val_loader)
    scheduler.step(val_f1)

    history['tr_acc'].append(tr_acc)
    history['val_f1'].append(val_f1)
    history['val_mae'].append(val_mae)
    history['val_r2'].append(val_r2)
    history['phase'].append(phase)

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        no_improve   = 0
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, OUT_DIR / 'pqfilm_dual_best.pt')
    else:
        no_improve += 1

    if ep % 5 == 0 or ep == 1:
        print(f'{ep:4d}  {phase:2d}  {tr_acc:.4f}  {val_f1:.4f}  {val_mae:.4f}  {val_r2:.4f}  {best_val_f1:.4f}  {no_improve}/{PATIENCE}')

    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {ep}')
        break

print(f'Best val F1: {best_val_f1:.4f}')

max=100 epochs  warmup=10  patience=20  bs=256
  Ep  Ph    TrAcc    ValF1     MAE      R2     Best  Wait


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep_range = range(1, len(history['tr_acc'])+1)
sw = next((i+1 for i, p in enumerate(history['phase']) if p==2), None)
for ax in axes:
    if sw: ax.axvline(sw, color='gray', ls='--', alpha=0.5, label='QNN frozen')
axes[0].plot(ep_range, history['tr_acc'], label='Train Acc')
axes[0].plot(ep_range, history['val_f1'], label='Val F1')
axes[0].set_title('Accuracy & F1'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(ep_range, history['val_mae'], color='orange')
axes[1].set_title('Val MAE (gram)'); axes[1].set_ylabel('MAE'); axes[1].grid(True)
axes[2].plot(ep_range, history['val_r2'], color='green')
axes[2].set_title('Val R²'); axes[2].set_ylabel('R²'); axes[2].grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR/'training_curves.png', dpi=200); plt.show()

In [ ]:
model.load_state_dict(best_weights)
te_acc, te_f1, te_mae, te_r2, te_pred, te_true = evaluate(model, te_loader)

print('='*60)
print('FINAL EVALUATION — UNSEEN TEST SET')
print('='*60)
print(f'  Accuracy : {te_acc:.4f}')
print(f'  F1-macro : {te_f1:.4f}')
print(f'  MAE (g)  : {te_mae:.4f}')
print(f'  R²       : {te_r2:.4f}')
print()
print(classification_report(te_true, te_pred, target_names=LABEL_NAMES, digits=4))
cm = confusion_matrix(te_true, te_pred)
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(LABEL_NAMES, rotation=30, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Dual-Branch PQ-FiLM (Stack 4-class)')
plt.colorbar(im, ax=ax)
thresh = cm.max()/2
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j]>thresh else 'black')
plt.tight_layout()
plt.savefig(OUT_DIR/'confusion_matrix.png', dpi=200); plt.show()

In [ ]:
summary = {
    'model': 'DualBranchPQFiLMQNN',
    'best_params': best_params,
    'best_val_f1': float(best_val_f1),
    'test': {'acc': float(te_acc), 'f1': float(te_f1),
             'mae': float(te_mae), 'r2':  float(te_r2)},
    'classes': LABEL_NAMES,
}
with open(OUT_DIR/'results_summary.json','w') as f:
    json.dump(summary, f, indent=2)
print(f'Checkpoint : {OUT_DIR}/pqfilm_dual_best.pt')
print(f'Summary    : {OUT_DIR}/results_summary.json')